# Acceptance: calibration entry points built on the workflow pieces

**This is a sign-off page, not a tutorial.** It exercises roadmap step 30
(composability package `CP2`, findings `CX16`–`CX19`, `CX24`). `wf.optimize`
returns a resumable `wf.OptimizeRun` and takes any optimisation method you
write; `BayesianModel.find_map` is a one-start `wf.optimize`;
`BayesianModel.sample` is a model-bound `wf.run_mcmc` that takes a numpyro
kernel you built and seeds from `Candidates`; both return the run object.
Nothing here asserts; the automated checks live in
`tests/test_calib_composable.py`.

The model is chosen so the workflow has real work to do: **many quite
different parameter sets fit the data almost equally well**, and they disagree
about what happens next. Every section is plotted on that likelihood surface.

Run every cell top to bottom (about three minutes — sections D and E compile
and run NUTS on a genuinely awkward posterior). Sign off only if every **What to check** box holds.

| | You are checking that… |
|---|---|
| A | multi-start optimisation finds genuinely different good fits, and a run continues exactly where it stopped |
| B | an optimisation method you write plugs into `wf.optimize` |
| C | `find_map` is exactly a one-start `wf.optimize`, and where it lands depends on its start |
| D | `sample` uses the numpyro kernel you built and is exactly `wf.run_mcmc` |
| E | seeded sampling explores the whole ridge, until a stop rule decides |
| F | `posterior_runs` takes the run objects, and shows how far the good fits disagree |

## The model: a ridge of good fits

An SIR epidemic in a population of one million with frequency-dependent
transmission. Two rates are unknown: the **contact rate** (infection) and the
**recovery rate**. The data are infectious counts every four days over the
first four weeks only — the growth phase — with negative-binomial noise.

Early growth is exponential at rate *contact − recovery*, so the data pin down
that **difference** but hardly the two rates separately. A slow epidemic with
low contact and slow recovery and a fast one with high contact and fast
recovery produce nearly the same first four weeks. The plot shows the log
density over both rates: a long diagonal ridge, with the truth (star) on it.

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import optax
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from numpyro.infer import NUTS

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from summer4 import (
    Compartments,
    FlowModel,
    OutputSet,
    Param,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    Target,
    TargetSet,
    TransitionFlow,
)
from summer4.epi import FOIKind, ForceOfInfection, MixingMatrix
from summer4.epi.calibration import BayesianModel, NegativeBinomial, Scenario, Uniform
from summer4.epi.calibration import workflow as wf

state = Property("state", ("S", "I", "R"))
pop = Property("pop", ("all",))
pmap = PropertyMap.from_property(state).stratify(pop)
model = FlowModel(pmap)
model.add_flow(
    TransitionFlow(
        "infection",
        state["S"],
        state["I"],
        ForceOfInfection(
            "infection",
            infectious=state["I"],
            group_by=pop,
            mixing=MixingMatrix(pop, np.array([[1.0]])),
            kind=FOIKind.FREQUENCY,
            contact_rate=Param("contact"),
        ),
    )
)
model.add_flow(TransitionFlow("recovery", state["I"], state["R"], Param("recovery")))
cm = model.compile()

N = 1.0e6
y0_values = np.zeros(pmap.size)
y0_values[pmap.select(state["S"])] = N - 20
y0_values[pmap.select(state["I"])] = 20
y0 = PropertyData.wrap(pmap, y0_values)
infectious = Compartments(where=state["I"])
run_kwargs = dict(t0=0.0, t1=180.0, dt=0.5, solver="euler")
TRUTH = {"contact": 0.45, "recovery": 0.2}

times = np.arange(0.0, 29.0, 4.0)
clean = cm.run(TRUTH, y0, save=SavePlan(requests={"I": SaveRequest(infectious, ts=times)}), **run_kwargs)
expected = np.asarray(clean["I"].values.data).reshape(-1)
observed = np.random.default_rng(3).negative_binomial(20, 20 / (20 + expected)).astype(float)

bm = BayesianModel(
    cm,
    {},
    priors=(Uniform("contact", 0.1, 1.5), Uniform("recovery", 0.02, 1.0)),
    targets=TargetSet(targets=(Target(key="I", times=times, values=observed, quantity=infectious, likelihood=NegativeBinomial(dispersion=20.0)),)),
    y0=y0,
    run_kwargs=run_kwargs,
)

# The log density on a grid, for every plot below.
contact_grid = np.linspace(0.1, 1.5, 30)
recovery_grid = np.linspace(0.02, 1.0, 30)
C, R = np.meshgrid(contact_grid, recovery_grid)
surface = wf.evaluate(bm, wf.Candidates.from_params(bm, {"contact": C.ravel(), "recovery": R.ravel()}), batch_size=300)
log_density = np.asarray(surface.log_density).reshape(C.shape)
best_on_grid = float(log_density.max())


def ridge_figure(title):
    fig = go.Figure(
        go.Contour(
            x=contact_grid,
            y=recovery_grid,
            z=np.clip(log_density, best_on_grid - 30, None),
            colorscale="Viridis",
            contours={"coloring": "heatmap", "showlines": False},
            colorbar={"title": "log density"},
            name="log density",
        )
    )
    fig.add_scatter(x=[TRUTH["contact"]], y=[TRUTH["recovery"]], mode="markers", marker={"symbol": "star", "size": 16, "color": "gold", "line": {"width": 1}}, name="truth")
    fig.update_layout(title=title, xaxis_title="contact rate", yaxis_title="recovery rate", height=560, legend={"orientation": "h", "y": -0.2})
    return fig


def trajectory(params):
    res = cm.run({k: float(v) for k, v in params.items()}, y0, save=SavePlan(requests={"I": SaveRequest(infectious)}), **run_kwargs)
    return pd.Series(np.asarray(res["I"].values.data).reshape(-1), index=np.asarray(res["I"].times.values))


fig = ridge_figure("The log density: a diagonal ridge of good fits")
fig.show()
pd.Series(observed, index=pd.Index(times, name="day"), name="observed infectious").plot(kind="bar", title="The data: the first four weeks only").update_layout(xaxis_title="day", yaxis_title="infectious", showlegend=False).show()

> **What to check**
>
> - The surface shows a long bright-yellow diagonal band running from about (0.3, 0.1) to (1.2, 1.0), not a single spot.
> - The star (truth) lies on the band.

## A. Multi-start finds genuinely different good fits — and resumes

Score a 64-point Latin hypercube, keep the eight best as starts, and optimise
them all with `wf.optimize` (Adam with its automatic learning-rate probe and
convergence). Lines join each start (circle) to where it ended (cross). The
table gives each end point's log density, its R0 (contact ÷ recovery) and the
epidemic it projects beyond the data.

In [ ]:
design = wf.evaluate(bm, wf.lhs(bm, 64, seed=0), batch_size=64)
starts = design.best(8)
multi = wf.optimize(bm, starts, chunk_steps=50, max_steps=300, seed=0)
ends = multi.candidates

fig = ridge_figure("A. Eight starts and where optimisation took them")
for i in range(len(starts)):
    fig.add_scatter(
        x=[starts.params["contact"][i], ends.params["contact"][i]],
        y=[starts.params["recovery"][i], ends.params["recovery"][i]],
        mode="lines+markers",
        marker={"symbol": ["circle-open", "x"], "size": 10},
        name=f"start {i}",
    )
fig.show()

rows = []
for i in range(len(ends)):
    params = {k: ends.params[k][i] for k in ends.sites}
    path = trajectory(params)
    rows.append(
        {
            "contact": float(params["contact"]),
            "recovery": float(params["recovery"]),
            "log density": float(ends.log_density[i]),
            "R0": float(params["contact"] / params["recovery"]),
            "growth rate": float(params["contact"] - params["recovery"]),
            "projected peak (people)": float(path.max()),
            "projected peak day": float(path.idxmax()),
        }
    )
table_a = pd.DataFrame(rows).sort_values("log density", ascending=False).round(3)
table_a

> **What to check**
>
> - Every cross sits on the bright band, and the crosses are **spread along it** — not piled on one spot.
> - In the table, every log density is within about 2–3 of the best, and every growth rate is close to 0.25 — yet R0 and the projected peak differ a lot (the peak by a factor of two or more). These are genuinely different answers that the data cannot tell apart.

Now the resume check. Four starts, plain Adam, convergence switched off so
every start runs every step: one run of 200 steps, and one of 100 steps
followed by `run.extend(100)`. If `extend` continues from the stored optimiser
state, the two loss traces are the same curve.

In [ ]:
adam = wf.Optax(learning_rate=0.05, plateau=False)
never_converge = wf.AutoTune(lr_grid=(0.05,), patience=10**6)
kw = dict(method=adam, tuning=never_converge, chunk_steps=50, seed=0)
four = starts.take(np.arange(4))

one_run = wf.optimize(bm, four, max_steps=200, **kw)
first = wf.optimize(bm, four, max_steps=100, **kw)
chunks_before = first.loss_trace.shape[0]
resumed = first.extend(100)

steps = 50 * np.arange(1, 5)
curves = {}
for i in range(len(four)):
    curves[f"start {i}: one 200-step run"] = pd.Series(one_run.loss_trace[:, i], index=steps)
    curves[f"start {i}: 100 + extend(100)"] = pd.Series(resumed.loss_trace[:, i], index=steps)
fig = pd.DataFrame(curves).rename_axis("step").plot(markers=True, log_y=True, title="A. Best loss per start: one run vs a resumed run")
fig.update_layout(xaxis_title="optimiser steps", yaxis_title="best loss (-log density)", legend_title="")
fig.show()

pd.Series(
    {
        "chunks before extend": chunks_before,
        "chunks after extend": resumed.loss_trace.shape[0],
        "max |resumed - one run| (loss trace)": float(np.max(np.abs(resumed.loss_trace - one_run.loss_trace))),
        "max |resumed - one run| (end points)": float(np.max(np.abs(resumed.candidates.params["contact"] - one_run.candidates.params["contact"]))),
        "history stages": [record.stage for record in resumed.history],
    },
    name="A",
    dtype=object,
)

> **What to check**
>
> - Each start's two markers sit on top of each other at every step, including 150 and 200 after the `extend`.
> - The table shows 2 chunks before `extend` and 4 after, and both differences are 0 or at float32 rounding.
> - `history stages` lists the design stages (`lhs`, `evaluate`, …) followed by **two** `optimize` records.

## B. An optimisation method you write plugs in

A method is anything with `make(potential_fn, ...)` returning a backend with
`init(z0, key)` and `step(state)`. This one is gradient descent with gradient
clipping — about twenty lines — run by the same chunked, vmapped,
restart-aware `wf.optimize` as the built-ins, from the same eight starts.

In [ ]:
class ClippedGradientDescent:
    def __init__(self, step_size, max_norm=1.0):
        self.step_size = step_size
        self.max_norm = max_norm

    def make(self, potential_fn, *, learning_rate=None, sites=None, template_z=None):
        step_size, max_norm = self.step_size, self.max_norm

        class Backend:
            def init(self, z0, key):
                z = {k: jnp.asarray(v) for k, v in z0.items()}
                return {"z": z, "best_z": z, "best_loss": potential_fn(z)}

            def step(self, state):
                loss, grads = jax.value_and_grad(potential_fn)(state["z"])
                norm = jnp.sqrt(sum(jnp.sum(g**2) for g in jax.tree.leaves(grads)))
                scale = step_size * jnp.minimum(1.0, max_norm / (norm + 1e-12))
                z = jax.tree.map(lambda v, g: v - scale * g, state["z"], grads)
                better = loss < state["best_loss"]
                best_z = jax.tree.map(lambda cur, best: jnp.where(better, cur, best), state["z"], state["best_z"])
                return {"z": z, "best_z": best_z, "best_loss": jnp.where(better, loss, state["best_loss"])}, best_z, state["best_loss"]

        return Backend()


mine = wf.optimize(bm, starts, method=ClippedGradientDescent(0.05), tuning=never_converge, chunk_steps=50, max_steps=300)
trace = pd.DataFrame(mine.loss_trace, columns=[f"start {i}" for i in range(len(starts))], index=50 * np.arange(1, mine.loss_trace.shape[0] + 1))
fig = trace.rename_axis("step").plot(markers=True, log_y=True, title="B. Your clipped gradient descent, run by wf.optimize")
fig.update_layout(xaxis_title="optimiser steps", yaxis_title="best loss (-log density)", legend_title="")
fig.show()

fig = ridge_figure("B. Where your method took the eight starts")
fig.add_scatter(x=mine.candidates.params["contact"], y=mine.candidates.params["recovery"], mode="markers", marker={"symbol": "x", "size": 10, "color": "crimson"}, name="your method's end points")
fig.show()
pd.Series(
    {
        "isinstance(method, wf.OptimizeMethod)": isinstance(ClippedGradientDescent(1.0), wf.OptimizeMethod),
        "run.method_name": mine.method_name,
        "best log density (Adam in A)": round(float(np.max(ends.log_density)), 3),
        "best log density (your method)": round(float(np.max(mine.candidates.log_density)), 3),
    },
    name="B",
    dtype=object,
)

> **What to check**
>
> - Every start's loss falls, or stays flat once it has arrived.
> - The red crosses sit on the bright band.
> - The table shows `True`, `clippedgradientdescent`, and a best log density close to Adam's in section A.

## C. `find_map` is a one-start `wf.optimize` — and on a ridge, where it lands depends on the start

`find_map` returns the optimisation run; `best_params` is the MAP estimate.
The cell prints the lines it stands for and runs them by hand for one seed.
Then three seeds — three different random starting points — each run
`find_map` for 150 steps, and each run is then extended by 300 more. A start
whose loss has stopped improving is frozen as converged (so `extend` leaves it
where it is); on a flat ridge that happens at different places for different
starts. The single-start answer is whatever the start leads to, which is why
multi-start and sampling matter.

In [ ]:
doc = bm.find_map.__doc__
print(doc[doc.index("Exactly::"):])

z0 = bm.init_point(0)
start = wf.Candidates.from_z(bm, {k: jnp.asarray(v)[None] for k, v in z0.items()})
by_hand = wf.optimize(bm, start, method=wf.Optax(optax.adam(0.05)), max_steps=150, chunk_steps=50, seed=0)

fig = ridge_figure("C. find_map from three starting points, then extend(300)")
rows = {}
for seed in (0, 1, 2):
    start_point = {k: float(v) for k, v in bm.constrain(bm.init_point(seed)).items()}
    run = bm.find_map(steps=150, seed=seed)
    after_150 = dict(run.best_params)
    ld_150 = float(run.candidates.log_density[0])
    run.extend(300)
    after = run.best_params
    fig.add_scatter(
        x=[start_point["contact"], float(after_150["contact"]), float(after["contact"])],
        y=[start_point["recovery"], float(after_150["recovery"]), float(after["recovery"])],
        mode="lines+markers",
        marker={"symbol": ["circle-open", "diamond", "x"], "size": 11},
        name=f"seed {seed}: start → 150 steps → +300",
    )
    rows[f"seed {seed}"] = {
        "MAP after 150 steps (contact, recovery)": (round(float(after_150["contact"]), 3), round(float(after_150["recovery"]), 3)),
        "log density after 150": round(ld_150, 3),
        "MAP after extend(300)": (round(float(after["contact"]), 3), round(float(after["recovery"]), 3)),
        "log density after extend": round(float(run.candidates.log_density[0]), 3),
        "R0 after extend": round(float(after["contact"] / after["recovery"]), 3),
    }
fig.show()
print("seed 0 by hand, after 150 steps:", {k: round(float(v), 4) for k, v in by_hand.best_params.items()})
pd.DataFrame(rows).T

> **What to check**
>
> - The printed expansion is `init_point` → `Candidates.from_z` → `wf.optimize(..., method=wf.Optax(optimizer or optax.adam(0.05)), ...)`, and the by-hand seed-0 result equals the seed-0 row's 150-step MAP.
> - In the plot, all three runs end on the band, but at **different places along it**.
> - `extend` never lowers a run's log density: a run still improving moves further along the band, and a run already marked converged stays where it was (its two table rows match).
> - The three final R0 values differ clearly — a single MAP estimate would hide that.

## D. `sample` uses the kernel you built

Build a NUTS kernel yourself — `target_accept_prob=0.95`, which suits a narrow
ridge — and hand it to `sample`, seeded from the multi-start end points of
section A. The run object holds your kernel, and `wf.run_mcmc` (the lines
`sample` stands for) reproduces the draws exactly. A tiny run: this section
checks the plumbing; section E does the real sampling.

In [ ]:
seeds = ends.take(np.argsort(-np.asarray(ends.log_density))[:4])
kernel = NUTS(bm.numpyro_model(), target_accept_prob=0.95)
short = bm.sample(kernel, init=seeds, num_warmup=10, num_samples=10, num_chains=4, seed=7)
again = wf.run_mcmc(
    bm, seeds, make_kernel=lambda: NUTS(bm.numpyro_model(), target_accept_prob=0.95),
    num_chains=4, num_warmup=10, chunk_samples=10, stop=lambda row: (True, "num_samples"), seed=7,
)
pd.Series(
    {
        "type": type(short).__name__,
        "run.mcmc.sampler is your kernel": short.mcmc.sampler is kernel,
        "kernel target_accept_prob": kernel._target_accept_prob,
        "max |sample - run_mcmc| (contact)": float(np.max(np.abs(short.samples["contact"] - again.samples["contact"]))),
        "run.idata posterior shape": tuple(np.asarray(short.idata.posterior["contact"]).shape),
    },
    name="D",
    dtype=object,
)

> **What to check**
>
> - `type` is `MCMCRun`, `run.mcmc.sampler is your kernel` is **True**, and the kernel's `target_accept_prob` reads **0.95**.
> - `max |sample - run_mcmc|` is exactly **0.0**, and `run.idata` holds 4 chains × 10 draws.

## E. Seeded sampling explores the whole ridge, until a stop rule decides

Four NUTS chains seeded from four good fits, sampling in chunks of 100 draws
until `wf.StopRule(rhat=1.05, ess=100)` is met. The draws are plotted on the
ridge; the progress table is the run's record of each stop decision.

In [ ]:
sampled = bm.sample(
    NUTS(bm.numpyro_model(), target_accept_prob=0.95),
    init=seeds,
    num_chains=4,
    num_warmup=200,
    num_samples=100,
    stop=wf.StopRule(rhat=1.05, ess=100, max_samples=1500),
    seed=0,
)
fig = ridge_figure("E. Posterior draws from four seeded chains")
for c in range(4):
    fig.add_scatter(x=sampled.samples["contact"][c], y=sampled.samples["recovery"][c], mode="markers", marker={"size": 4, "opacity": 0.6}, name=f"chain {c}")
fig.show()

draws = pd.DataFrame({k: sampled.samples[k].reshape(-1) for k in ("contact", "recovery")})
draws["R0"] = draws["contact"] / draws["recovery"]
display_quantiles = draws.quantile([0.05, 0.5, 0.95]).round(3)
display_quantiles.index = ["5%", "median", "95%"]
print(display_quantiles)
sampled.progress

> **What to check**
>
> - The draws spread **along the band** — most of its length — and all four chains' colours are mixed together rather than each sitting in its own patch.
> - The printed R0 interval (5% to 95%) is wide, far wider than the spread of any single fit.
> - In the progress table, `decision` is empty on every row except the last, which reads `diagnostics`; there `rhat_max` ≤ 1.05 and `ess_bulk_min` ≥ 100, and on every earlier row at least one of the two fails.

## F. `posterior_runs` takes the run objects — and shows the disagreement

`posterior_runs` accepts the `MCMCRun` itself (its draws) and the
`OptimizeRun` itself (its best points). The same draws passed as the run and as
`run.idata` give identical runs. The fit only ever saved the data days; a
`Scenario` with its own `OutputSet` asks the forward runs for infectious people
over all six months. The ribbon is the posterior's 95% band; the dotted lines
are the eight multi-start fits from section A; the dots are the data.

In [ ]:
projection = OutputSet()
projection["infectious"] = Compartments(where=state["I"]).total()
six_months = {"projection": Scenario(outputs=projection)}

from_run = bm.posterior_runs(sampled, n=80, seed=1, batch_size=40, scenarios=six_months)
from_idata = bm.posterior_runs(sampled.idata, n=80, seed=1, batch_size=40, scenarios=six_months)
from_optimum = bm.posterior_runs(multi, n=None, batch_size=8, scenarios=six_months)

ribbon = from_run.quantiles(q=(0.025, 0.5, 0.975))["projection"]
band = pd.DataFrame(
    {
        "2.5%": ribbon[("infectious", "0.025")].to_numpy(),
        "median": ribbon[("infectious", "0.5")].to_numpy(),
        "97.5%": ribbon[("infectious", "0.975")].to_numpy(),
    },
    index=np.asarray(ribbon.index),
).rename_axis("day")
fig = band.plot(title="F. Posterior 95% band, the eight multi-start fits, and the data")
for i, path in enumerate(from_optimum.samples("projection", "infectious")):
    fig.add_scatter(x=band.index, y=np.asarray(path).reshape(-1), mode="lines", line={"dash": "dot", "width": 1}, name=f"fit {i}")
fig.add_scatter(x=times, y=observed, mode="markers", marker={"size": 8, "color": "black"}, name="data")
fig.add_vline(x=28, line_dash="dash", annotation_text="end of data")
fig.update_layout(xaxis_title="day", yaxis_title="infectious", legend_title="")
fig.show()

pd.Series(
    {
        "max |from run - from idata|": float(np.max(np.abs(from_run.samples("projection", "infectious") - from_idata.samples("projection", "infectious")))),
        "multi-start trajectories": from_optimum.samples("projection", "infectious").shape[0],
        "band width at day 28": round(float(band["97.5%"].loc[28.0] - band["2.5%"].loc[28.0])),
        "band width at its widest": round(float((band["97.5%"] - band["2.5%"]).max())),
    },
    name="F",
    dtype=object,
)

> **What to check**
>
> - Up to the dashed *end of data* line the band is narrow and passes through the data; after it the band fans out widely.
> - The dotted multi-start fits agree up to day 28 and then peak at clearly different heights and times.
> - `max |from run - from idata|` is exactly **0.0**, there are **8** multi-start trajectories, and the band at its widest is many times its width at day 28.